# IMPORTACIÓN DE DATOS Y LIBRERÍAS

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv(os.path.join('Datos', 'Transformados', 'df_meteorologico.csv'), index_col = 0)

In [3]:
nuevo = []
for i in range(df.shape[0]):
    if df['recurrence'].iloc[i] == 1:
        nuevo.append(True)
    elif df['recurrence'].iloc[i] > 1:
        nuevo.append(False)
    else:
        nuevo.append('REVISAR')

In [4]:
df['cliente_nuevo'] = nuevo
df = df.reset_index()

In [5]:
datos = df
var_meteo = ['MEAN_tmin', 'MEAN_tmax', 'MEAN_tmed', 'MEAN_sol', 'MEAN_velmedia', 'MEAN_racha', 'MEAN_hrMedia', 'MEAN_prec']
for var in var_meteo:
    df = df[~df[str(var)].isna()]

# LIMPIEZA RÁPIDA

In [6]:
df['completed_entry_forms_count'] = df['completed_entry_forms_count'].fillna(value = 0)

# SELECCIÓN DE VARIABLES PARA LA CLUSTERIZACIÓN

In [7]:
X = df

In [8]:
cols_rm = ['F_C_I', 'F_C_O', 'IDEMA', 'idema_code', 'region', 'city',
   'MIN_tmin', 'Q1_tmin', 'Q2_tmin', 'Q3_tmin', 'MAX_tmin', 'IQR_tmin', 'STD_tmin',
   'MIN_tmax', 'Q1_tmax', 'Q2_tmax', 'Q3_tmax', 'MAX_tmax', 'IQR_tmax', 'STD_tmax',
   'MIN_tmed', 'Q1_tmed', 'Q2_tmed', 'Q3_tmed', 'MAX_tmed', 'IQR_tmed', 'STD_tmed',
   'MIN_prec', 'Q1_prec', 'Q2_prec', 'Q3_prec', 'MAX_prec', 'IQR_prec', 'STD_prec',
   'MIN_sol', 'Q1_sol', 'Q2_sol', 'Q3_sol', 'MAX_sol', 'IQR_sol', 'STD_sol',
   'MIN_velmedia', 'Q1_velmedia', 'Q2_velmedia', 'Q3_velmedia', 'MAX_velmedia', 'IQR_velmedia', 'STD_velmedia',
   'MIN_racha', 'Q1_racha', 'Q2_racha', 'Q3_racha', 'MAX_racha', 'IQR_racha', 'STD_racha',
   'MIN_hrMedia', 'Q1_hrMedia', 'Q2_hrMedia', 'Q3_hrMedia', 'MAX_hrMedia', 'IQR_hrMedia', 'STD_hrMedia']
for var in cols_rm:
    del X[str(var)]

In [9]:
vars_fecha = ['booked_at', 'checkin_time', 'checkout_time', 'asset_opening_date', 'last_entry_form_completed_at', 'cancelled_at']
for var in vars_fecha:
    del X[str(var)]

In [10]:
vars_superfluas = ['brand', 'available_units', 'bought_products', 'rate', 'cancellation_lead_time', 'status', 'stay_length', 'requested_category_name', 'travel_agency_name']
for var in vars_superfluas:
    del X[str(var)]

In [11]:
var_bool = ['all_entry_forms_completed', 'returning_inhabitant', 'libere_community']
mapping = {'yes': 1, 'no': 0}
for var in var_bool:
    X[str(var)] = X[str(var)].map(mapping)

In [12]:
var_boolean = var_bool + ['is_cancelled']
for var in var_boolean:
    X[str(var)] = X[str(var)].astype('bool')

In [13]:
var_pista = ['all_entry_forms_completed', 'completed_entry_forms_count', 'recurrence','libere_community','returning_inhabitant', 'cancellation_reason', 'is_cancelled', 'days_before_cancel', 'cliente_nuevo']
for var in var_pista:
    del X[str(var)]

In [14]:
for var in var_meteo:
    del X[str(var)]

# FORMATEO DE LAS VARIABLES SELECCIONADAS PARA LA CLUSTERIZACION

In [15]:
X['checkin_month'] = X['checkin_month'].map({1: 'ENE', 2: 'FEB', 3: 'MAR', 
                        4: 'ABR', 5: 'MAY', 6: 'JUN', 
                        7: 'JUL', 8: 'AGO', 9: 'SEP', 
                        10: 'OCT', 11: 'NOV', 12: 'DIC'})
X['checkin_day'] = X['checkin_day'].map({1: 'LUN', 2: 'MAR', 3: 'MIE', 4: 'JUE', 5: 'VIE', 6: 'SAB', 7: 'DOM'})

In [16]:
X['index'] = X['index'].astype('category')

In [17]:
var_num = X.dtypes.reset_index()[X.dtypes.reset_index()[0] == 'int64']['index'].to_list() + X.dtypes.reset_index()[X.dtypes.reset_index()[0] == 'float64']['index'].to_list()
var_cat = X.dtypes.reset_index()[X.dtypes.reset_index()[0] == 'object']['index'].to_list()
var_bool = X.dtypes.reset_index()[X.dtypes.reset_index()[0] == 'bool']['index'].to_list()

In [18]:
medias = []
desves = []
for var in var_num:
    medias.append(float(X[str(var)].mean()))
    desves.append(float(X[str(var)].std()))
normalizadores = pd.DataFrame({'MU': medias, 'SIGMA': desves, 'VAR': var_num})

In [19]:
for var in var_num:
    normalizer = StandardScaler()
    X[str(var)] = normalizer.fit_transform(X[[str(var)]])

In [20]:
X = pd.get_dummies(X, columns = var_cat)

In [21]:
for i in X.columns.to_list():
    if X[i].isna().sum() != 0:
        print(i)

OUTLIERS

In [22]:
var_num = X.dtypes.reset_index()[X.dtypes.reset_index()[0] == 'int64']['index'].to_list() + X.dtypes.reset_index()[X.dtypes.reset_index()[0] == 'float64']['index'].to_list()
outliers_bool = pd.DataFrame()
for var in var_num:
    outliers_bool[str(var)] = (X[str(var)] > 2.7)
outliers_na = X[var_num][~outliers_bool]
index_outliers = []
for i in range(outliers_na.shape[0]):
    if outliers_na.iloc[i].isna().sum() != 0:
        index_outliers.append(i)
X = X.drop(X.index[index_outliers]).reset_index()

In [23]:
Y = X.copy()

In [24]:
del X['level_0']
del X['index']

# KMEANS

In [25]:
from sklearn.metrics import silhouette_samples, silhouette_score, make_scorer
from sklearn.cluster import KMeans
from sklearn.model_selection import GridSearchCV

In [26]:
kmeans = KMeans(random_state = 4) #MODELO BASE, CON LOS PARAMETROS FIJOS

#PARÁMETROS PARA BUSCAR EL MEJOR MODELO
param_grid_kmeans = {
    'n_clusters': [3,4,5], #Número de clústers, literalmente el K de Kmeans
    'init': ['k-means++', 'random'], #Método para inicializar los centroides
    'n_init': [10,20] #Número de veces que el algoritmo se ejecuta con diferentes inicializaciones. Luego, se guarda el resultado con menos inercia (el mejor).
}

#SELECCIÓN DE LA MÉTRICA DE ÉXITO
scorer = make_scorer(silhouette_score)

#CREACIÓN GRIDSEARCH
grid = GridSearchCV(
    estimator = kmeans,
    param_grid = param_grid_kmeans,
    scoring = scorer,
    cv = 2, #NO TIENE SENTIDO HACER CROSS VALIDATION, PERO GRIDSEARCH LO EXIGE
    n_jobs = -1
)

In [27]:
# HAY SOSPECHA DE QUE GRID SEARCH NO FUNCIONE CORRECTAMENTE CON KMEANS

# import warnings
# warnings.filterwarnings('ignore')

# #AJUSTE DE GRIDSEARCH A LOS DATOS
# grid.fit(X)

# #MEJORES PARÁMETROS E INDICE DE SILUETA
# print("Mejores parámetros:", grid.best_params_)
# print("Mejor score:", grid.best_score_)

# #MEJOR MODELO Y RESULTADOS
# best_kmeans = grid.best_estimator_
# labels = best_kmeans.labels_
# best_kmeans_params = best_kmeans.get_params()
# kmeans = KMeans(algorithm = best_kmeans_params['algorithm'],
#                 copy_x = best_kmeans_params['copy_x'],
#                 init = best_kmeans_params['init'], 
#                 max_iter = best_kmeans_params['max_iter'],
#                 n_clusters = best_kmeans_params['n_clusters'],
#                 n_init = best_kmeans_params['n_init'],
#                 random_state = best_kmeans_params['random_state'],
#                 tol = best_kmeans_params['tol'],
#                 verbose = best_kmeans_params['verbose'])
# clusters_kmeans = kmeans.fit_predict(X)
# silhouette_avg_kmeans = silhouette_score(X, clusters_kmeans)
# centroides_kmeans = kmeans.cluster_centers_
# print('RESULTADO DE KMEANS:')
# print(f'Para {best_kmeans_params['n_clusters']} clusters, el índice de silueta es: {silhouette_avg_kmeans}')

In [ ]:
kmeans_silhouette = {}
for k in param_grid_kmeans['n_clusters']:
    for init in param_grid_kmeans['init']:
        for n_init in param_grid_kmeans['n_init']:
            kmeans = KMeans(n_clusters = k, 
                            random_state = 4, 
                            init = init,
                            n_init = n_init)
            clusters_kmeans = kmeans.fit_predict(X)
            key = f'{k}_{init}_{n_init}'
            kmeans_silhouette[key] = silhouette_score(X, clusters_kmeans)
            print(f'Kmeans {key} hecho.')
dict_to_df = {
    'MODELO': list(kmeans_silhouette.keys()),
    'IND_SILUETA': list(kmeans_silhouette.values())
}
kmeans_silhouette = pd.DataFrame(dict_to_df)
print(f'El modelo con mejor índice de silueta es: {kmeans_silhouette[kmeans_silhouette['IND_SILUETA'].max() == kmeans_silhouette['IND_SILUETA']]['MODELO'].to_list()[0]} y el índice de silueta es de {kmeans_silhouette[kmeans_silhouette['IND_SILUETA'].max() == kmeans_silhouette['IND_SILUETA']]['IND_SILUETA'].to_list()[0]}')

Kmeans 3_k-means++_10 hecho.
Kmeans 3_k-means++_20 hecho.


# CLUSTERIZACIÓN JERÁRQUICA

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram

In [ ]:
def plot_dendrogram(model, **kwargs):
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count
    linkage_matrix = np.column_stack([model.children_, model.distances_, counts]).astype(float)
    dendrogram(linkage_matrix, **kwargs)

In [ ]:
hclust = AgglomerativeClustering()

param_grid_hclust_1 = {
    'n_clusters': [3, 4, 5],
    'metric': ['euclidean', 'l1', 'l2', 'manhattan'],
    'linkage': ['complete', 'average', 'single']
}

param_grid_hclust_2 = {
    'n_clusters': [3, 4, 5],
    'metric': ['euclidean'],
    'linkage': ['ward']
}

scorer = make_scorer(silhouette_score)

grid_hclust_1 = GridSearchCV(
    estimator = hclust,
    param_grid = param_grid_hclust_1,
    scoring = scorer,
    cv = 2
)

grid_hclust_2 = GridSearchCV(
    estimator = hclust,
    param_grid = param_grid_hclust_2,
    scoring = scorer,
    cv = 2
)

In [ ]:
#TARDA MUCHO EN EJECUTARSE, POR ELLO SE DESCARTA

# grid_hclust_1.fit(X)
# print("Mejores parámetros:", grid_hclust_1.best_params_)
# print("Mejor score:", grid_hclust_1.best_score_)
# grid_hclust_2.fit(X)
# print("Mejores parámetros:", grid_hclust_2.best_params_)
# print("Mejor score:", grid_hclust_2.best_score_)

# DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_samples, silhouette_score, make_scorer

In [ ]:
eps_values = [1.5]
min_samples_values = [40]
metric_options = ['manhattan', 'euclidean']

best_score = -1
best_params = {}

for dist in metric_options:
    for eps in eps_values:
        for min_samples in min_samples_values:
            dbscan = DBSCAN(
                eps=eps,
                min_samples=min_samples,
                metric=dist
            )
            
            labels = dbscan.fit_predict(X)

            mask = labels != -1
            labels_no_noise = labels[mask]
            X_no_noise = X[mask]

            if len(set(labels_no_noise)) < 2:
                continue

            score = silhouette_score(X_no_noise, labels_no_noise)

            if score > best_score:
                best_score = score
                best_params = {
                    'eps': eps,
                    'min_samples': min_samples,
                    'metric': dist
                }

best_dbscan = DBSCAN(**best_params)
labels = best_dbscan.fit_predict(X)

clusters1 = pd.DataFrame({'CLUSTERS':(labels.tolist())})

print("Mejores parámetros:", best_params)
print("Mejor Silhouette Score:", best_score)
print(f'Distribución de {clusters1.value_counts().to_list()}')

Mejores parámetros: {'eps': 1.5, 'min_samples': 40, 'metric': 'manhattan'}
Mejor Silhouette Score: 0.6101684642654088
Distribución de [43129, 63, 63, 53, 52, 52, 52, 52, 50, 49, 48, 47, 47, 45, 45, 43, 43, 41, 41, 40]


In [ ]:
Y['cluster'] = labels

In [ ]:
X = X.iloc[clusters1[clusters1['CLUSTERS'] == -1].index]
Y = Y.iloc[clusters1[clusters1['CLUSTERS'] == -1].index]

In [ ]:
eps_values = [1.5]
min_samples_values = [40]
metric_options = ['euclidean', 'manhattan']

best_score = -1
best_params = {}

for dist in metric_options:
    for eps in eps_values:
        for min_samples in min_samples_values:
            dbscan = DBSCAN(
                eps=eps,
                min_samples=min_samples,
                metric=dist
            )
            
            labels = dbscan.fit_predict(X)

            mask = labels != -1
            labels_no_noise = labels[mask]
            X_no_noise = X[mask]

            if len(set(labels_no_noise)) < 2:
                continue

            score = silhouette_score(X_no_noise, labels_no_noise)

            if score > best_score:
                best_score = score
                best_params = {
                    'eps': eps,
                    'min_samples': min_samples,
                    'metric': dist
                }

best_dbscan = DBSCAN(**best_params)
labels = best_dbscan.fit_predict(X)

clusters1 = pd.DataFrame({'CLUSTERS':(labels.tolist())})

print("Mejores parámetros:", best_params)
print("Mejor Silhouette Score:", best_score)
print(f'Distribución de {clusters1.value_counts().to_list()}')

Mejores parámetros: {'eps': 1.5, 'min_samples': 40, 'metric': 'euclidean'}
Mejor Silhouette Score: 0.21353299108490986
Distribución de [30742, 3222, 2611, 2595, 2133, 1025, 264, 127, 112, 97, 67, 56, 42, 36]


In [ ]:
del Y['cluster']
Y['cluster'] = labels

# RECUPERACIÓN DE LOS DATOS ORIGINALES

In [ ]:
df_clust = Y.copy()

In [ ]:
for var in normalizadores['VAR'].to_list():
    mu = list(normalizadores[normalizadores['VAR'] == var]['MU'])[0]
    sigma = list(normalizadores[normalizadores['VAR'] == var]['SIGMA'])[0]
    df_clust[str(var)] = np.round((df_clust[str(var)]*sigma+mu), decimals = 2)

In [ ]:
var_month = ['checkin_month_ENE', 'checkin_month_FEB', 'checkin_month_MAR', 'checkin_month_ABR', 'checkin_month_MAY',
 'checkin_month_JUN', 'checkin_month_JUL', 'checkin_month_AGO', 'checkin_month_SEP', 'checkin_month_OCT', 
 'checkin_month_NOV','checkin_month_DIC']
mes = []
for i in range(Y.shape[0]):
    instancia = Y[var_month].iloc[i]
    mes.append(instancia.index[instancia].to_list()[0][-3:])
for var in var_month:
    del df_clust[str(var)]
df_clust['checkin_month'] = mes

In [ ]:
var_week = ['checkin_day_DOM', 'checkin_day_LUN',
       'checkin_day_MAR', 'checkin_day_MIE', 'checkin_day_JUE',
       'checkin_day_VIE', 'checkin_day_SAB']
weekday = []
for i in range(Y.shape[0]):
    instancia = Y[var_week].iloc[i]
    weekday.append(instancia.index[instancia].to_list()[0][-3:])
for var in var_week:
    del df_clust[str(var)]
df_clust['checkin_day'] = weekday

In [ ]:
var_origin = ['origin_channel_manager', 'origin_direct_channel', 'origin_email', 'origin_in_person', 'origin_telephone']
origin = []
for i in range(Y.shape[0]):
    origin.append((df_clust[var_origin].iloc[i].index[df_clust[var_origin].iloc[i]].to_list()[0]).split(sep = '_')[-1])
for var in var_origin:
    del df_clust[str(var)]
df_clust['origin'] = origin

In [ ]:
var_requested = ['requested_category_hostel bed', 'requested_category_hostel room',
       'requested_category_one bedroom apartment', 'requested_category_one bedroom apartment superior',
       'requested_category_studio', 'requested_category_studio superior',
       'requested_category_three bedroom apartment', 'requested_category_three bedroom apartment superior',
       'requested_category_two bedroom apartment', 'requested_category_two bedroom apartment superior']
req = []
for i in range(Y.shape[0]):
    req.append((df_clust[var_requested].iloc[i].index[df_clust[var_requested].iloc[i]].to_list()[0]).split(sep = '_')[-1])
for var in var_requested:
    del df_clust[str(var)]
df_clust['requested_category'] = req

In [ ]:
var_asset = ['asset_Koisi Hostel',
       'asset_Líbere Bilbao La Vieja', 'asset_Líbere Bilbao Museo',
       'asset_Líbere Córdoba Patio Santa Marta',
       'asset_Líbere Granada Catedral', 'asset_Líbere Madrid Palacio Real',
       'asset_Líbere Málaga Teatro Romano', 'asset_Líbere Málaga la Merced',
       'asset_Líbere Pamplona Yamaguchi', 'asset_Líbere Valencia Abastos',
       'asset_Líbere Valencia Jardín Botánico', 'asset_Líbere Vitoria']
asset = []
for i in range(Y.shape[0]):
    asset.append((df_clust[var_asset].iloc[i].index[df_clust[var_asset].iloc[i]].to_list()[0]).split(sep = '_')[-1])
for var in var_asset:
    del df_clust[str(var)]
df_clust['asset'] = asset

In [ ]:
var_as_type = ['asset_type_aparthotel', 'asset_type_apartment', 'asset_type_hostel']
asset = []
for i in range(Y.shape[0]):
    asset.append((df_clust[var_as_type].iloc[i].index[df_clust[var_as_type].iloc[i]].to_list()[0]).split(sep = '_')[-1])
for var in var_as_type:
    del df_clust[str(var)]
df_clust['asset_type'] = asset

In [ ]:
var_bs = ['business_segment_business group',
       'business_segment_business individual',
       'business_segment_leisure group', 'business_segment_leisure individual',
       'business_segment_mice group', 'business_segment_mice individual',
       'business_segment_mid stay', 'business_segment_travel group',
       'business_segment_travel individual']
asset = []
for i in range(Y.shape[0]):
    asset.append((df_clust[var_bs].iloc[i].index[df_clust[var_bs].iloc[i]].to_list()[0]).split(sep = '_')[-1])
for var in var_bs:
    del df_clust[str(var)]
df_clust['business_segment'] = asset

In [ ]:
var_rtg = ['rate_group_name_Flexible | Groups | Basic',
       'rate_group_name_Flexible | Groups | Best',
       'rate_group_name_Flexible | Groups | Super',
       'rate_group_name_Flexible | Travel Trade, Corporate & MICE',
       'rate_group_name_Flexible-2d | B2C',
       'rate_group_name_Flexible-30d | B2C',
       'rate_group_name_Flexible-7d | B2C', 'rate_group_name_Mid Stay',
       'rate_group_name_Non Refundable | B2C',
       'rate_group_name_Non Refundable | Travel Trade, Corporate & MICE']
asset = []
for i in range(Y.shape[0]):
    asset.append((df_clust[var_rtg].iloc[i].index[df_clust[var_rtg].iloc[i]].to_list()[0]).split(sep = '_')[-1])
for var in var_rtg:
    del df_clust[str(var)]
df_clust['rate_group_name'] = asset

KeyboardInterrupt: 

In [ ]:
var_rate_type = ['rate_type_Flexible', 'rate_type_Mid Stay',
       'rate_type_Non Refundable']
asset = []
for i in range(Y.shape[0]):
    asset.append((df_clust[var_rate_type].iloc[i].index[df_clust[var_rate_type].iloc[i]].to_list()[0]).split(sep = '_')[-1])
for var in var_rate_type:
    del df_clust[str(var)]
df_clust['rate_type'] = asset

In [ ]:
var_names_original = datos.columns.to_list()
var_names = df_clust.columns.to_list()
var_names_rest = set(var_names_original) - set(var_names)
df_to_join = datos[list(var_names_rest)].iloc[df_clust['index']]
df_to_join2 = df_to_join.reset_index()
df_merged = pd.merge(left = df_clust, right = df_to_join2, how = 'left', on = 'index')
del df_merged['index']
del df_merged['level_0']

In [ ]:
var_names_original[0] = 'cluster'

In [ ]:
df_final = df_merged[var_names_original]

# PERFIL DE LOS SEGMENTOS

In [ ]:
cluster_names = (df_final['cluster'].value_counts().index[df_final['cluster'].value_counts() > 1000]).to_list()

In [ ]:
clus = pd.DataFrame()
for clu in cluster_names:
    new_clus = df_final['cluster'] == clu
    clus = pd.concat([clus, new_clus.astype('int')], axis = 1)

In [ ]:
clus_fila = []
for i in range(df_final.shape[0]):
   clus_fila.append(int(clus.iloc[i].sum()))

In [ ]:
df_final['clus'] = clus_fila
df_final['clus'] = df_final['clus'].map({1: True, 0: False})
df_final = df_final[df_final['clus']]
df_final = df_final.reset_index()
del df_final['index']
del df_final['clus']

In [ ]:
cluster_names

In [ ]:
df_final['cluster'].map({-1:1,0:})

,index,cluster,booked_at,checkin_time,checkout_time,lead_time,lenght_of_stay,checkin_month,checkin_day,adult_count,...,MIN_hrMedia,Q1_hrMedia,Q2_hrMedia,Q3_hrMedia,MAX_hrMedia,IQR_hrMedia,MEAN_hrMedia,STD_hrMedia,cliente_nuevo,clus
0,0,0,2022-11-26 16:10:00,2023-01-01 12:00:00,2023-01-02 12:00:00,36.0,1.0,ENE,DOM,1.0,...,30.0,30.00,30.0,30.00,30.0,0.0,30.0,0.0,False,True
1,1,-1,2022-09-24 12:09:00,2023-01-01 15:00:00,2023-01-02 12:00:00,99.0,1.0,ENE,DOM,2.0,...,30.0,30.00,30.0,30.00,30.0,0.0,30.0,0.0,True,True
2,2,-1,2022-10-18 07:12:00,2023-01-01 15:00:00,2023-01-02 12:00:00,75.0,1.0,ENE,DOM,4.0,...,30.0,30.00,30.0,30.00,30.0,0.0,30.0,0.0,True,True
3,3,-1,2022-11-25 17:50:00,2023-01-01 15:00:00,2023-01-02 12:00:00,37.0,1.0,ENE,DOM,2.0,...,30.0,30.00,30.0,30.00,30.0,0.0,30.0,0.0,True,True
4,4,-1,2022-11-26 04:10:00,2023-01-01 15:00:00,2023-01-03 12:00:00,36.0,2.0,ENE,DOM,3.0,...,30.0,43.75,57.5,71.25,85.0,27.5,57.5,27.5,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42323,43124,-1,2023-10-10 20:53:00,2023-12-31 15:00:00,2024-01-01 11:00:00,82.0,1.0,DIC,DOM,4.0,...,76.0,76.00,76.0,76.00,76.0,0.0,76.0,0.0,True,True
42324,43125,-1,2023-10-10 20:53:00,2023-12-31 15:00:00,2024-01-01 11:00:00,82.0,1.0,DIC,DOM,4.0,...,76.0,76.00,76.0,76.00,76.0,0.0,76.0,0.0,True,True
42325,43126,-1,2023-10-13 13:38:00,2023-12-31 15:00:00,2024-01-01 11:00:00,79.0,1.0,DIC,DOM,4.0,...,76.0,76.00,76.0,76.00,76.0,0.0,76.0,0.0,True,True
42326,43127,-1,2023-10-13 13:38:00,2023-12-31 15:00:00,2024-01-01 11:00:00,79.0,1.0,DIC,DOM,4.0,...,76.0,76.00,76.0,76.00,76.0,0.0,76.0,0.0,True,True
